In [7]:
# !pip install openai python-dotenv pandas
import pandas as pd
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import textwrap

import truststore
truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

MODEL = 'gpt-4o-mini'




client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

API key loaded successfully.
OpenAI client ready.


In [8]:
products = [
    {"id": "P001", "name": "Wireless Bluetooth Headphones", "description": "Over-ear wireless headphones with active noise cancellation, 30-hour battery life, and foldable design. Features Bluetooth 5.3 and multipoint connectivity."},
    {"id": "P002", "name": "Ergonomic Office Chair", "description": "Adjustable lumbar support office chair with breathable mesh back, 4D armrests, seat depth adjustment, and 135-degree recline. Supports up to 300 lbs."},
    {"id": "P003", "name": "Stainless Steel Water Bottle", "description": "Double-wall vacuum insulated 1-liter water bottle. Keeps drinks cold 24 hours or hot 12 hours. BPA-free, leak-proof lid with carrying loop."},
    {"id": "P004", "name": "Mechanical Keyboard", "description": "Compact 75% layout mechanical keyboard with hot-swappable switches, RGB backlighting, gasket mount structure, and PBT double-shot keycaps. USB-C and wireless."},
    {"id": "P005", "name": "Portable Power Station", "description": "500Wh portable power station with LiFePO4 battery, 800W pure sine wave AC output, 100W USB-C PD charging, and solar panel input. Weighs 14 lbs."},
    {"id": "P006", "name": "Smart Doorbell Camera", "description": "2K HDR video doorbell with 180-degree field of view, two-way audio, night vision, and local storage via microSD. Works with Alexa and Google Home."},
    {"id": "P007", "name": "Air Purifier", "description": "HEPA 13 air purifier covering 1,200 sq ft. Auto mode with air quality sensor, whisper-quiet 24dB sleep mode, and filter replacement indicator."},
    {"id": "P008", "name": "Camping Hammock", "description": "Ultralight double camping hammock made from 70D ripstop nylon. Holds 500 lbs. Includes tree straps, carabiners, and stuff sack. Packs down to 1 lb."},
    {"id": "P009", "name": "Electric Standing Desk", "description": "Dual-motor electric standing desk with 60x30 inch bamboo top. Height range 25.6-51.2 inches, 4 memory presets, cable management tray, and anti-collision sensor."},
    {"id": "P010", "name": "Robot Vacuum Cleaner", "description": "LiDAR navigation robot vacuum with 5000Pa suction, auto-empty dock, mop attachment, no-go zones via app, and 180-minute runtime on hardwood and carpet."}
]

df = pd.DataFrame(products)
df

,id,name,description
0,P001,Wireless Bluetooth Headphones,Over-ear wireless headphones with active noise...
1,P002,Ergonomic Office Chair,Adjustable lumbar support office chair with br...
2,P003,Stainless Steel Water Bottle,Double-wall vacuum insulated 1-liter water bot...
3,P004,Mechanical Keyboard,Compact 75% layout mechanical keyboard with ho...
4,P005,Portable Power Station,500Wh portable power station with LiFePO4 batt...
5,P006,Smart Doorbell Camera,2K HDR video doorbell with 180-degree field of...
6,P007,Air Purifier,"HEPA 13 air purifier covering 1,200 sq ft. Aut..."
7,P008,Camping Hammock,Ultralight double camping hammock made from 70...
8,P009,Electric Standing Desk,Dual-motor electric standing desk with 60x30 i...
9,P010,Robot Vacuum Cleaner,LiDAR navigation robot vacuum with 5000Pa suct...


In [9]:
short_prompt = "You are an assistant that summarizes products for SEO and extracts key features. Return JSON with keys: summary (2 sentences) and features (list of 3-5 strings)."

In [10]:
naive_results = []

for product in products[:5]:  # start with 5 products
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions=short_prompt,
        input=f"Product: {product['name']}\nDescription: {product['description']}"
    )

    elapsed = time.time() - start
    
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens if usage.input_tokens_details else 0

    naive_results.append({
        "product": product["name"],
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cached_tokens": cached,
        "latency_s": round(elapsed, 2),
        "output_preview": response.output_text[:120] + "..."
    })
    
    print(f"✅ {product['name']}: {elapsed:.2f}s | input={usage.input_tokens} | cached={cached} | output={usage.output_tokens}")

print(f"\n⏱️  Total time: {sum(r['latency_s'] for r in naive_results):.2f}s")

✅ Wireless Bluetooth Headphones: 11.51s | input=86 | cached=0 | output=98
✅ Ergonomic Office Chair: 9.31s | input=89 | cached=0 | output=100
✅ Stainless Steel Water Bottle: 9.37s | input=87 | cached=0 | output=109
✅ Mechanical Keyboard: 3.44s | input=87 | cached=0 | output=117
✅ Portable Power Station: 3.91s | input=96 | cached=0 | output=124

⏱️  Total time: 37.54s



Prompt caching is **automatic** — no code changes required. Here's the mechanism:

```
Request 1:  [=== system prompt (1500 tokens) ===][user: Product A]
                     ↓ computed & cached

Request 2:  [=== system prompt (1500 tokens) ===][user: Product B]
                     ↓ cache HIT! skip recomputation

Request 3:  [=== system prompt (1500 tokens) ===][user: Product C]
                     ↓ cache HIT! 
```

In [11]:
long_system_prompt = """
You are an expert e-commerce product content specialist working for a major online marketplace.
Your job is to create SEO-optimized product summaries and extract structured feature data.

=== OUTPUT FORMAT ===
You MUST respond with valid JSON only. No markdown, no explanation, just the JSON object.
{
  "seo_summary": "A compelling 2-3 sentence summary optimized for search engines. Include the product name, primary benefit, and a call-to-action phrase. Use active voice throughout.",
  "features": ["Feature 1 as a concise bullet point", "Feature 2", ...],
  "category": "The most specific product category",
  "target_audience": "Who this product is best suited for",
  "keywords": ["seo", "keyword", "list"]
}

=== WRITING STYLE GUIDE ===
1. Always write in active voice. BAD: "The battery is designed to last 30 hours." GOOD: "Enjoy 30 hours of uninterrupted battery life."
2. Lead with the primary benefit, not the feature. BAD: "Has Bluetooth 5.3" GOOD: "Connect seamlessly to all your devices with Bluetooth 5.3"
3. Use power words: discover, transform, elevate, essential, premium, revolutionary, seamless
4. Avoid superlatives without evidence. Don't say "best" or "greatest" unless the product description provides comparative data.
5. Keep sentences between 15-25 words for readability.
6. Include at least one emotional trigger word per summary (comfort, confidence, peace of mind, freedom, etc.)
7. End the SEO summary with an implicit call-to-action ("Upgrade your...", "Transform your...", "Experience...")

=== FEATURE EXTRACTION RULES ===
1. Extract 3-5 features maximum. Prioritize by consumer impact.
2. Each feature should be a standalone phrase (not a full sentence). 5-12 words each.
3. Always include the primary spec (battery life, weight, capacity, etc.) with units.
4. Group related specs into a single feature when possible.
5. Order features from most important to least important for the target consumer.

=== CATEGORY TAXONOMY ===
Use the most specific category from this hierarchy:
- Electronics > Audio > Headphones > Over-Ear / In-Ear / On-Ear
- Electronics > Computers > Peripherals > Keyboards / Mice / Monitors
- Electronics > Smart Home > Security / Lighting / Climate / Cleaning
- Furniture > Office > Chairs / Desks / Storage
- Outdoor & Sports > Camping > Shelter / Sleep / Cook / Accessories
- Kitchen & Home > Drinkware / Cookware / Appliances
- Power & Energy > Portable Power / Solar / Batteries

=== KEYWORD GUIDELINES ===
1. Generate 5-8 SEO keywords per product.
2. Include both short-tail (1-2 words) and long-tail (3-5 words) keywords.
3. Always include the product type, primary feature, and target use case.
4. Avoid brand names unless provided in the product description.
5. Include common misspellings or alternative terms consumers might search for.

=== FEW-SHOT EXAMPLES ===

EXAMPLE INPUT:
Product: Running Shoes
Description: Lightweight mesh running shoes with responsive foam cushioning, reinforced heel counter, and reflective details. Available in sizes 6-14. Weighs 8.5 oz.

EXAMPLE OUTPUT:
{
  "seo_summary": "Hit the pavement with confidence in these ultralight running shoes featuring responsive foam cushioning that absorbs impact mile after mile. The breathable mesh upper and reinforced heel deliver the perfect balance of comfort and stability. Elevate your running experience today.",
  "features": ["Responsive foam cushioning for impact absorption", "Breathable lightweight mesh upper at 8.5 oz", "Reinforced heel counter for stability", "Reflective details for low-light visibility", "Available in sizes 6 through 14"],
  "category": "Outdoor & Sports > Running > Footwear",
  "target_audience": "Recreational and intermediate runners seeking lightweight, cushioned footwear",
  "keywords": ["running shoes", "lightweight running shoes", "cushioned running shoes", "mesh running shoes", "reflective running shoes", "responsive foam shoes", "8.5 oz running shoes"]
}

EXAMPLE INPUT:
Product: Yoga Mat
Description: 6mm thick non-slip yoga mat made from natural tree rubber with alignment markings. Dimensions 72x26 inches. Includes carrying strap. Free from PVC, latex, and silicone.

EXAMPLE OUTPUT:
{
  "seo_summary": "Transform your practice on this premium natural rubber yoga mat with built-in alignment markings that guide your poses with precision. The 6mm thick non-slip surface provides the perfect cushion-to-grip ratio for any flow style. Discover your best practice with this eco-friendly essential.",
  "features": ["6mm natural tree rubber with non-slip grip", "Built-in alignment markings for guided practice", "72x26 inch full-size dimensions", "Eco-friendly: PVC, latex, and silicone free", "Includes carrying strap for portability"],
  "category": "Outdoor & Sports > Yoga > Mats",
  "target_audience": "Yoga practitioners of all levels seeking an eco-conscious, non-slip mat",
  "keywords": ["yoga mat", "non-slip yoga mat", "natural rubber yoga mat", "alignment yoga mat", "eco-friendly yoga mat", "6mm yoga mat", "yoga mat with markings"]
}

=== IMPORTANT REMINDERS ===
- Output ONLY the JSON object. No preamble, no markdown code fences, no explanation.
- If the product description is vague, make reasonable inferences but do not fabricate specific numbers.
- Use American English spelling throughout.
- Ensure all JSON keys are exactly as shown in the format specification above.
"""

# Rough token count check (1 token ≈ 4 chars for English)
estimated_tokens = len(long_system_prompt) / 4
print(f"Estimated system prompt tokens: ~{int(estimated_tokens)}")
print(f"Meets 1024-token minimum: {'✅ Yes' if estimated_tokens >= 1024 else '❌ No — need to expand'}")

Estimated system prompt tokens: ~1344
Meets 1024-token minimum: ✅ Yes


In [14]:
cache_results = []

for i, product in enumerate(products[:5]):
    start = time.time()
    
    response = client.responses.create(
        model=MODEL,
        instructions=long_system_prompt,
        input=f"Product: {product['name']}\nDescription: {product['description']}"
    )
    
    elapsed = time.time() - start
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens if usage.input_tokens_details else 0
    
    cache_results.append({
        "request_num": i + 1,
        "product": product["name"],
        "input_tokens": usage.input_tokens,
        "cached_tokens": cached,
        "cache_hit_pct": f"{(cached / usage.input_tokens * 100):.1f}%" if usage.input_tokens > 0 else "0%",
        "output_tokens": usage.output_tokens,
        "latency_s": round(elapsed, 2)
    })
    
    print(f"Request {i+1}: {product['name'][:30]:30s} | latency={elapsed:.2f}s | cached={cached}/{usage.input_tokens} tokens")

print(f"\n⏱️  Total: {sum(r['latency_s'] for r in cache_results):.2f}s")

Request 1: Wireless Bluetooth Headphones  | latency=6.87s | cached=0/1250 tokens
Request 2: Ergonomic Office Chair         | latency=7.82s | cached=0/1253 tokens
Request 3: Stainless Steel Water Bottle   | latency=3.98s | cached=1024/1251 tokens
Request 4: Mechanical Keyboard            | latency=5.97s | cached=1024/1251 tokens
Request 5: Portable Power Station         | latency=3.91s | cached=1024/1260 tokens

⏱️  Total: 28.55s


Caching is best-effort, openai routes all the reqeusts to machines based on some hash, so it's not necessary your request will always hit the cache, but in general you can expect a very high cache hit rate for repeated system prompts.

In [15]:
# First, send one more request with the ORIGINAL prompt to warm the cache
response_warm = client.responses.create(
    model=MODEL,
    instructions=long_system_prompt,
    input=f"Product: {products[5]['name']}\nDescription: {products[5]['description']}"
)
cached_warm = response_warm.usage.input_tokens_details.cached_tokens if response_warm.usage.input_tokens_details else 0
print(f"Original prompt  → cached: {cached_warm}/{response_warm.usage.input_tokens}")

# Now modify ONE WORD in the system prompt
modified_prompt = long_system_prompt.replace("expert e-commerce", "skilled e-commerce")  # just one word changed!

response_miss = client.responses.create(
    model=MODEL,
    instructions=modified_prompt,
    input=f"Product: {products[5]['name']}\nDescription: {products[5]['description']}"
)
cached_miss = response_miss.usage.input_tokens_details.cached_tokens if response_miss.usage.input_tokens_details else 0
print(f"Modified prompt   → cached: {cached_miss}/{response_miss.usage.input_tokens}")
print(f"\n💡 Changing one word in the prefix caused a cache {'MISS ❌' if cached_miss < cached_warm else 'HIT ✅'}")

Original prompt  → cached: 0/1255
Modified prompt   → cached: 0/1255

💡 Changing one word in the prefix caused a cache HIT ✅


In [17]:
INPUT_COST_PER_M = 0.05
CACHED_COST_PER_M = 0.005
OUTPUT_COST_PER_M = 0.40


def calculate_cost_from_averages(
    num_requests,
    avg_input_tokens,
    avg_cached_tokens,
    avg_output_tokens,
):
    if avg_cached_tokens > avg_input_tokens:
        raise ValueError("avg_cached_tokens cannot exceed avg_input_tokens")

    total_input = num_requests * avg_input_tokens
    total_cached = num_requests * avg_cached_tokens
    total_uncached = total_input - total_cached
    total_output = num_requests * avg_output_tokens

    input_cost = (
        (total_uncached / 1_000_000) * INPUT_COST_PER_M
        + (total_cached / 1_000_000) * CACHED_COST_PER_M
    )
    output_cost = (total_output / 1_000_000) * OUTPUT_COST_PER_M
    total_cost = input_cost + output_cost

    return {
        "num_requests": num_requests,
        "total_input_tokens": total_input,
        "cached_tokens": total_cached,
        "uncached_tokens": total_uncached,
        "output_tokens": total_output,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


def compare_scenarios(num_requests, naive, cached):
    naive_cost = calculate_cost_from_averages(num_requests, **naive)
    cached_cost = calculate_cost_from_averages(num_requests, **cached)

    savings = naive_cost["total_cost"] - cached_cost["total_cost"]
    savings_pct = (savings / naive_cost["total_cost"]) * 100 if naive_cost["total_cost"] else 0

    return naive_cost, cached_cost, savings, savings_pct


NUM_REQUESTS = 1_000_000

naive_profile = {
    "avg_input_tokens": 4000,
    "avg_cached_tokens": 0,
    "avg_output_tokens": 100,
}

cached_profile = {
    "avg_input_tokens": 4000,
    "avg_cached_tokens": 3500,
    "avg_output_tokens": 100,
}

naive_cost, cached_cost, savings, savings_pct = compare_scenarios(
    NUM_REQUESTS,
    naive_profile,
    cached_profile,
)

print("Naive total cost:  $%.2f" % naive_cost["total_cost"])
print("Cached total cost: $%.2f" % cached_cost["total_cost"])
print("Savings:           $%.2f (%.2f%%)" % (savings, savings_pct))

Naive total cost:  $240.00
Cached total cost: $82.50
Savings:           $157.50 (65.62%)
